# Bars from a tick stream: `resample` and string aggregations

Raw market data arrives as a *tick stream*: one row per trade, each with a
price, a volume, and a timestamp. `resample` condenses that stream into
fixed-interval **bars** — the first step in almost any bar-based feature
pipeline.

This notebook covers the simplest way to drive it: the `every=` bucket width
and the built-in **string aggregations**. We start with a single-column `mean`
bar, then build the standard OHLCV candle, then split volume into buyer- and
seller-initiated flow with `ohlcv2`.

Every bar is **causal**: its value depends only on ticks already seen inside
the bar, never on future ones. A bucket is emitted only once a later timestamp
proves it complete; the trailing partial bucket is flushed at end of input.

> Later in this series: custom per-bar statistics with functor and dict `agg`
> (next), `count=` and sparse-clock bars, and running the same bar definition
> inside a `Dag` so batch and live give identical results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer.streams import resample

rng = np.random.default_rng(42)
n = 80

# Irregular integer timestamps: each tick lands 1-2 units after the last.
timestamps = np.cumsum(rng.integers(1, 3, size=n)).astype(np.int64)

# Mid-price: a slow random walk around 100.
price = 100.0 + np.cumsum(rng.standard_normal(n) * 0.3)

# Trade size: heavy-tailed, always positive.
volume = rng.exponential(scale=50.0, size=n)

# Signed volume: + = buyer-initiated, - = seller-initiated (size drawn
# independently of sign, a deliberately simple microstructure model).
signed_volume = volume * rng.choice([-1.0, 1.0], size=n)

pd.DataFrame({
    "timestamp":     timestamps[:6],
    "price":         price[:6].round(3),
    "volume":        volume[:6].round(1),
    "signed_volume": signed_volume[:6].round(1),
}).set_index("timestamp")

## The simplest bar: one column, one reducer

Pass a 1-D value array, its `index` (the timestamps), and a bucket width
`every=`. With a string `agg` like `"mean"`, each bar reduces to a single
number.

The return value is a `Stream`. It unpacks as `(values, index)` for
convenience, and single-column results expose `.values` and `.index` directly.
Other single-column shorthands work the same way: `first`, `last`, `min`,
`max`, `sum`, `count`.

In [ ]:
BAR = 20

bars = resample(price, timestamps, every=BAR, agg="mean")

# A Stream unpacks as (values, index):
values, index = bars
print("bar starts:", index)
print("bar means: ", values.round(3))

## OHLCV candles

For price bars, pass a two-column `[price, volume]` array and `agg="ohlcv"`.
Each bar yields five **named** columns — `open`, `high`, `low`, `close`,
`volume` — collected causally from the ticks inside it.

Multi-column results set `.columns`, so you can read a single series by name
(`bars["close"]`) or take the whole array with `.values`.

In [ ]:
ohlcv = resample(np.column_stack([price, volume]), timestamps,
                 every=BAR, agg="ohlcv")

print("columns:", ohlcv.columns)
pd.DataFrame(
    ohlcv.values,
    index=pd.Index(ohlcv.index, name="bar_open"),
    columns=list(ohlcv.columns),
).round(3)

In [ ]:
# Candlestick view: OHLC range on top, bar volume below.
fig, (ax_price, ax_vol) = plt.subplots(
    2, 1, figsize=(9, 5), sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)

for t, row in zip(ohlcv.index, ohlcv.values):
    o, h, l, c = row[:4]
    color = "steelblue" if c >= o else "crimson"
    ax_price.plot([t, t], [l, h], color="0.4", lw=1.5, zorder=1)   # high-low wick
    ax_price.bar(t, c - o, bottom=o, width=BAR * 0.5,              # open-close body
                 color=color, alpha=0.8, zorder=2)

ax_price.set_ylabel("price")
ax_price.set_title(f"OHLCV bars  (every={BAR})")

ax_vol.bar(ohlcv.index, ohlcv["volume"], width=BAR * 0.5, color="0.6")
ax_vol.set_ylabel("volume")
ax_vol.set_xlabel("bar open index")
plt.tight_layout()

## Buyer- vs seller-initiated volume: `ohlcv2`

`agg="ohlcv2"` takes the same two-column input but reads column 1 as *signed*
volume. It returns the four OHLC columns plus `buy_vol` and `sell_vol`:

- `buy_vol`  — sum of the positive part of signed volume per bar,
- `sell_vol` — sum of the magnitude of the negative part per bar.

That's the signed-part decomposition `x = PosPart(x) - NegPart(x)` applied to
order flow. Their difference is *net flow* — positive when buyers dominate the
bar.

In [ ]:
ohlcv2 = resample(np.column_stack([price, signed_volume]), timestamps,
                  every=BAR, agg="ohlcv2")

print("columns:", ohlcv2.columns)
net_flow = ohlcv2["buy_vol"] - ohlcv2["sell_vol"]

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(ohlcv2.index,  ohlcv2["buy_vol"],  width=BAR * 0.5, color="seagreen", label="buy")
ax.bar(ohlcv2.index, -ohlcv2["sell_vol"], width=BAR * 0.5, color="crimson",  label="sell")
ax.plot(ohlcv2.index, net_flow, color="0.2", lw=1.5, marker="o", ms=4, label="net flow")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("bar open index")
ax.set_ylabel("signed volume")
ax.set_title("Buyer- vs seller-initiated volume per bar")
ax.legend(fontsize=9)
plt.tight_layout()

## Recap and what's next

`resample(values, index, every=, agg=...)` turns a tick stream into causal
bars. With **string aggregations** you get one-liners for the common cases:
`mean`, `first` / `last` / `min` / `max` / `sum` / `count`, and the `ohlc` /
`ohlcv` / `ohlcv2` price-bar family. Results are labelled `Stream`s you index
by column name.

Next in the series:

- **Custom per-bar statistics** — pass a functor (e.g. `ExpandingSkew()`) or a
  dict of them to compute your own bar features.
- **Bucketing modes and sparse data** — `count=` bars and dense-clock handling
  for empty windows.
- **Bars in a `Dag`** — wire the same bar definition into a graph so batch and
  live runs give identical results.